In [1]:
import sys
 
file_path = 'randomfile.txt'
sys.stdout = open(file_path, "w")

In [6]:
!pip3 install selenium

In [7]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
import copy 

/Users/vakul/Documents/Projects/RPGOYAL EPROC Automation/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [13]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options

# Optional: set up Chrome options
chrome_options = Options()
chrome_options.add_argument("--start-maximized")
# chrome_options.add_argument("--headless")  # Uncomment if needed

# Automatically download the right ChromeDriver and create Service
service = Service(ChromeDriverManager().install())

# Start Chrome with the driver
driver = webdriver.Chrome(service=service, options=chrome_options)

# Go to website
driver.get('https://eproc.rajasthan.gov.in/')
time.sleep(2)


In [11]:
!pip3 install webdriver_manager

In [45]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager


def init_driver():
    options = Options()
    options.add_argument("--start-maximized")
    # options.add_argument("--headless")  # Uncomment if needed
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)


def navigate_to_main_page(driver):
    driver.get("https://eproc.rajasthan.gov.in/")
    time.sleep(2)
    driver.find_element(By.ID, "PageLink_10").click()
    time.sleep(2)


def get_organisations(driver):
    return driver.find_elements(By.XPATH, "//a[contains(@id,'DirectLink')]")


def get_tenders(driver):
    return driver.find_elements(By.XPATH, "//a[contains(@id,'DirectLink')]")


def get_field_text_from_popup(driver, field_name):
    """
    Extract field text from inside the popup modal.
    Adjust xpath if popup structure differs.
    """
    try:
        xpath = f"//td/b[normalize-space()='{field_name}']/ancestor::td/following-sibling::td[1]"
        element = driver.find_element(By.XPATH, xpath)
        return element.text.strip()
    except NoSuchElementException:
        return "Not Found"


def handle_popup_and_extract_fields(driver, fields_to_extract):
    """
    Wait for popup, extract fields, close popup.
    """

    try:
        # Wait for popup/modal container to appear
        popup = WebDriverWait(driver, 10).until(
            EC.visibility_of_element_located((By.ID, "popup_content"))  # <-- Replace with actual popup modal ID
        )
        # Extract fields inside popup
        tender_data = {}
        for field in fields_to_extract:
            tender_data[field] = get_field_text_from_popup(driver, field)

        # Close popup: Find close button inside popup and click it
        close_btn = popup.find_element(By.CSS_SELECTOR, ".popup_close")  # <-- Replace with actual close button selector
        close_btn.click()

        # Wait until popup disappears
        WebDriverWait(driver, 5).until(EC.invisibility_of_element(popup))

        return tender_data

    except TimeoutException:
        print("Popup did not appear or timed out")
        return None
    except Exception as e:
        print(f"Error handling popup: {e}")
        return None


def main():
    driver = init_driver()
    navigate_to_main_page(driver)

    organisations = get_organisations(driver)
    num_orgs = len(organisations)

    for i in range(num_orgs):
        if i == 0:
            org_element = driver.find_element(By.ID, "DirectLink")
        else:
            org_element = driver.find_element(By.ID, f"DirectLink_{i-1}")

        org_element.click()
        time.sleep(2)

        tenders = get_tenders(driver)
        tender_links = [t.get_attribute("href") for t in tenders]

        fields_to_extract = [
            "Title",
            "Work Description",
            "Tender Value in ₹",
            "Location",
            "Organisation Chain",
            "Bid Submission End Date"
        ]

        for j in range(len(tender_links)):

            # Navigate to Organisation again (page reload)
            if i == 0:
                driver.find_element(By.ID, "DirectLink").click()
            else:
                driver.find_element(By.ID, f"DirectLink_{i-1}").click()
            time.sleep(2)

            # # Click the tender link to open popup/modal
            # if j == 0:
            #     driver.find_element(By.ID, "DirectLink").click()
            # else:
            #     driver.find_element(By.ID, f"DirectLink_{j-1}").click()

            # Handle popup: Extract info, then close it
            # tender_data = handle_popup_and_extract_fields(driver, fields_to_extract)

            tender_data = {}
            for field in fields_to_extract:
                tender_data[field] = get_field_text_from_popup(driver, field)
            if tender_data:
                for key, value in tender_data.items():
                    print(f"{key}: {value}")
            else:
                print(f"Failed to extract popup info for tender {j}")

            time.sleep(1)  # small pause before next tender

        break  # remove if you want to loop through all orgs

    driver.quit()


if __name__ == "__main__":
    main()


In [100]:

def get_org_tenders(org_name, org_id):
    driver = init_driver()
    navigate_to_main_page(driver)

    organisations = get_organisations(driver)
    num_orgs = len(organisations)
    org_element = driver.find_element(By.ID, org_id)

    org_element.click()

    tenders = get_tenders(driver)

    tender_links = [t.get_attribute("href") for t in tenders]

    fields_to_extract = [
                "Title",
                "Work Description",
                "Tender Value in ₹",
                "Location",
                "Organisation Chain",
                "Bid Submission End Date"
            ]
    
    tenders = []
    for i in tender_links:
        tender_data = {}
        driver.get(i)
        tender_data['Organisation'] = org_name
        tender_data['Tender Link'] = i
        for field in fields_to_extract:
            tender_data[field] = get_field_text_from_popup(driver, field)
            time.sleep(1)
        tenders.append(tender_data)
    return tenders

In [98]:
(org_name, org_id) = ("JDA", "DirectLink_32")

driver = init_driver()
navigate_to_main_page(driver)

organisations = get_organisations(driver)
num_orgs = len(organisations)
org_element = driver.find_element(By.ID, org_id)

org_element.click()

tenders = get_tenders(driver)

tender_links = [t.get_attribute("href") for t in tenders]



In [99]:
tender_links

['https://eproc.rajasthan.gov.in/nicgep/app?component=%24DirectLink&page=FrontEndViewTender&service=direct&session=T&sp=S1pbB%2FHaQs1TPwiyd4Ik4Ug%3D%3D',
 'https://eproc.rajasthan.gov.in/nicgep/app?component=%24DirectLink&page=FrontEndViewTender&service=direct&session=T&sp=S8cDwoPNv8wLQWwWn5smQaQ%3D%3D',
 'https://eproc.rajasthan.gov.in/nicgep/app?component=%24DirectLink&page=FrontEndViewTender&service=direct&session=T&sp=S5%2B5dC9uCSqpFO4eeY%2Fio%2BQ%3D%3D',
 'https://eproc.rajasthan.gov.in/nicgep/app?component=%24DirectLink&page=FrontEndViewTender&service=direct&session=T&sp=Sajgv7u9N%2FoVPn3YZLmndUQ%3D%3D',
 'https://eproc.rajasthan.gov.in/nicgep/app?component=%24DirectLink&page=FrontEndViewTender&service=direct&session=T&sp=SVC5fRQtE2fJRkGlb8eacdA%3D%3D',
 'https://eproc.rajasthan.gov.in/nicgep/app?component=%24DirectLink&page=FrontEndViewTender&service=direct&session=T&sp=SEcXRmymZEllq2CDgkAKAsA%3D%3D',
 'https://eproc.rajasthan.gov.in/nicgep/app?component=%24DirectLink&page=Front

In [101]:
tenders = get_org_tenders("JDA", "DirectLink_32")

In [91]:
!pip3 install pandas

In [92]:
import pandas as pd

In [102]:
df = pd.DataFrame(tenders)

In [105]:
df.to_csv('jda_tenders.csv', index=False)

In [96]:
df['Tender Link'][0]

'https://eproc.rajasthan.gov.in/nicgep/app?component=%24DirectLink&page=FrontEndViewTender&service=direct&session=T&sp=S7tYXeXNe6kE8C%2BpEOx8x1w%3D%3D'

In [97]:
df['Tender Link'][1]

'https://eproc.rajasthan.gov.in/nicgep/app?component=%24DirectLink&page=FrontEndViewTender&service=direct&session=T&sp=S7tYXeXNe6kE8C%2BpEOx8x1w%3D%3D'

In [94]:
tenders

[{'Organisation': 'JDA',
  'Tender Link': 'https://eproc.rajasthan.gov.in/nicgep/app?component=%24DirectLink&page=FrontEndViewTender&service=direct&session=T&sp=S7tYXeXNe6kE8C%2BpEOx8x1w%3D%3D',
  'Title': 'Preparation of GIS based Master Development Plan - 2047 for Jaipur Region',
  'Work Description': 'Preparation of GIS based Master Development Plan - 2047 for Jaipur Region',
  'Tender Value in ₹': '21,50,00,000',
  'Location': 'EE TA to DE 1',
  'Organisation Chain': 'Jaipur Development Authority||Director - Engineering - II||S.E. - 02||Ex. En. and TA to Dir.Eng.- II',
  'Bid Submission End Date': '30-May-2025 06:00 PM'},
 {'Organisation': 'JDA',
  'Tender Link': 'https://eproc.rajasthan.gov.in/nicgep/app?component=%24DirectLink&page=FrontEndViewTender&service=direct&session=T&sp=S7tYXeXNe6kE8C%2BpEOx8x1w%3D%3D',
  'Title': 'Preparation of GIS based Master Development Plan - 2047 for Jaipur Region',
  'Work Description': 'Preparation of GIS based Master Development Plan - 2047 for 

In [63]:
tender_data = {}
tender_data['Organisation'] = org_name
for field in fields_to_extract:
    tender_data[field] = get_field_text_from_popup(driver, field)

In [64]:
tender_data

{'Organisation': 'Rajasthsan',
 'Title': 'Civil work NIT02/2025-26/02',
 'Work Description': 'Lift irrigation structure and electric motor pump set at MAF, Ummedganj, Kota',
 'Tender Value in ₹': '8,00,000',
 'Location': 'Ummedganj, Kota',
 'Organisation Chain': 'Agriculture University Kota||Vice Chancellor',
 'Bid Submission End Date': '22-May-2025 05:00 PM'}

In [57]:
tender_data

{'Title': 'Civil work NIT02/2025-26/02',
 'Work Description': 'Lift irrigation structure and electric motor pump set at MAF, Ummedganj, Kota',
 'Tender Value in ₹': '8,00,000',
 'Location': 'Ummedganj, Kota',
 'Organisation Chain': 'Agriculture University Kota||Vice Chancellor',
 'Bid Submission End Date': '22-May-2025 05:00 PM'}

In [ ]:
def get_field_text_from_popup(driver, field_name):
    """
    Extract field text from inside the popup modal.
    Adjust xpath if popup structure differs.
    """
    try:
        xpath = f"//td/b[normalize-space()='{field_name}']/ancestor::td/following-sibling::td[1]"
        element = driver.find_element(By.XPATH, xpath)
        return element.text.strip()
    except NoSuchElementException:
        return "Not Found"

In [18]:
driver.quit()
driver = webdriver.Chrome('/Users/vakulgoyal/Downloads/chromedriver')
driver.get('https://eproc.rajasthan.gov.in/')
time.sleep(2) # Let the user actually see something!

Organistion_button = driver.find_element_by_id('PageLink_12')
Organistion_button.click()
Single_Organisation = driver.find_element_by_id('DirectLink_53')
Single_Organisation.click()
Tenders_List = driver.find_elements(By.XPATH,"//a[contains(@id,'DirectLink')]")
Tenderl = []
for k in Tenders_List:
    Tenderl.append(k.get_attribute('href'))
Tender_n = len(Tenders_List)
print(Tender_n)
time.sleep(2)
for j in range(0,Tender_n-1):
    if j==0:
        driver.quit()
        driver = webdriver.Chrome('/Users/vakulgoyal/Downloads/chromedriver')
        driver.get('https://eproc.rajasthan.gov.in/')
        time.sleep(2) # Let the user actually see something!

        Organistion_button = driver.find_element_by_id('PageLink_12')
        Organistion_button.click()
        Single_Organisation = driver.find_element_by_id('DirectLink_53')
        Single_Organisation.click()
        Tender_file = driver.find_element_by_id('DirectLink')
        Tender_file.click()
        Tender_Price = driver.find_element(By.XPATH,"(//table[contains(@class,'tablebg')])[6]/tbody[1]/tr[5]/td[2]")
        print("PRICE:" + Tender_Price.text)
        Tender_Category = driver.find_element(By.XPATH,"(//table[contains(@class,'tablebg')])[6]/tbody[1]/tr[5]/td[4]")
        print("Category : " + Tender_Category.text)
        Tender_Location = driver.find_element(By.XPATH,"(//table[contains(@class,'tablebg')])[6]/tbody[1]/tr[7]/td[2]")
        print("Location:" + Tender_Location.text)
        print("Link :" + Tenderl[j])
        print("*********************")
        print(str(j)+" i am if" )

    else:
        driver.quit()
        driver = webdriver.Chrome('/Users/vakulgoyal/Downloads/chromedriver')
        driver.get('https://eproc.rajasthan.gov.in/')
        time.sleep(2) # Let the user actually see something!

        Organistion_button = driver.find_element_by_id('PageLink_12')
        Organistion_button.click()
        Single_Organisation = driver.find_element_by_id('DirectLink_53')
        Single_Organisation.click()
        Tender_file = driver.find_element_by_id('DirectLink_'+str(j-1))
        Tender_file.click()
        Tender_Price = driver.find_element(By.XPATH,"(//table[contains(@class,'tablebg')])[6]/tbody[1]/tr[5]/td[2]")
        print("PRICE:" + Tender_Price.text)
        Tender_Category = driver.find_element(By.XPATH,"(//table[contains(@class,'tablebg')])[6]/tbody[1]/tr[5]/td[4]")
        print("Category : " + Tender_Category.text)
        Tender_Location = driver.find_element(By.XPATH,"(//table[contains(@class,'tablebg')])[6]/tbody[1]/tr[7]/td[2]")
        print("Location:" + Tender_Location.text)
        print("Link :" + Tenderl[j])
        print("*********************")
        print(str(j)+" i am else" )
            # driver.back()
time.sleep(5)

AttributeError: 'str' object has no attribute 'capabilities'

In [30]:
for i in range(0,Tender_n):
    print(i)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
